# JobFit — Phase 3 : Score ML supervisé

Feature engineering + entraînement de modèles + tracking MLflow.

In [ ]:
import sys, os
sys.path.append('../src/ml')
sys.path.append('../src/nlp')
sys.path.append('../src/api')

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from cv_parser  import parse_cv
from embeddings import EmbeddingEngine
from scorer     import build_features, generate_pseudo_labels, train_scorer, score_offres

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('../mlruns', exist_ok=True)

DATA_DIR  = '../data/processed'
CV_PATH   = '../data/cv/sample_cv.pdf'
MODELS_DIR = '../models'

plt.style.use('seaborn-v0_8-whitegrid')
print('Setup OK.')

## 1. Chargement CV + offres + embeddings

In [ ]:
# CV
cv_data = parse_cv(CV_PATH)
engine  = EmbeddingEngine()
cv_embedding = engine.embed_text(cv_data['enriched_text'])
print(f'CV : {cv_data["nb_competences"]} compétences')

# Offres
conn      = sqlite3.connect(f'{DATA_DIR}/jobfit.db')
df_offres = pd.read_sql('SELECT * FROM offres', conn)
conn.close()
print(f'Offres : {len(df_offres)}')

# Embeddings pré-calculés
offres_embeddings = np.load(f'{DATA_DIR}/offres_embeddings.npy')
print(f'Embeddings shape : {offres_embeddings.shape}')

## 2. Feature engineering

In [ ]:
print('Construction des features...')
features_df = build_features(cv_data, df_offres, cv_embedding, offres_embeddings)
print(f'Shape features : {features_df.shape}')
features_df.describe().round(3)

In [ ]:
# Visualiser les corrélations entre features
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(features_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True)
ax.set_title('Corrélations entre features', fontsize=13)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/features_correlation.png', dpi=150)
plt.show()

## 3. Génération des pseudo-labels

In [ ]:
labels = generate_pseudo_labels(features_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(labels, bins=30, color='steelblue', edgecolor='white')
ax.axvline(labels.mean(), color='red', linestyle='--', label=f'Moyenne: {labels.mean():.3f}')
ax.set_title('Distribution des pseudo-labels (scores de compatibilité)', fontsize=13)
ax.set_xlabel('Score [0-1]')
ax.legend()
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/labels_distribution.png', dpi=150)
plt.show()

print(f'Labels — min: {labels.min():.3f} | max: {labels.max():.3f} | mean: {labels.mean():.3f}')

## 4. Entraînement des modèles + tracking MLflow

In [ ]:
import mlflow
mlflow.set_tracking_uri('../mlruns')

import sys
sys.path.insert(0, '..')
import os
os.environ['MODELS_DIR']            = '../models'
os.environ['MLFLOW_TRACKING_URI']   = '../mlruns'

print('Entraînement des modèles (Ridge / RandomForest / GradientBoosting)...')
print('Résultats cross-validation 5-fold :\n')
best_model, results = train_scorer(features_df, labels, experiment_name='jobfit_scoring')

print('\nRésumé :')
pd.DataFrame(results).T

## 5. Scoring final + analyse

In [ ]:
df_scored = score_offres(best_model, features_df, df_offres)

print('--- TOP 15 OFFRES (score ML) ---')
cols = ['intitule', 'entreprise', 'lieu', 'type_contrat', 'ml_score_pct']
df_scored[cols].head(15)

In [ ]:
# Comparaison score cosinus vs score ML
df_compare = df_scored.copy()
df_compare['cosine_score_pct'] = (features_df['cosine_similarity'].values * 100)
df_compare = df_compare.sort_values('ml_score_pct', ascending=False).head(30)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(df_compare))
ax.plot(x, df_compare['ml_score_pct'].values,    'o-', label='Score ML',     color='steelblue', linewidth=2)
ax.plot(x, df_compare['cosine_score_pct'].values, 's--', label='Score cosinus', color='coral',     linewidth=1.5)
ax.set_title('Comparaison Score ML vs Score Cosinus (Top 30)', fontsize=13)
ax.set_xlabel('Rang')
ax.set_ylabel('Score (%)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/ml_vs_cosine.png', dpi=150)
plt.show()

In [ ]:
# Feature importance (si GradientBoosting ou RandomForest)
from sklearn.pipeline import Pipeline

model_step = best_model.named_steps['model']
if hasattr(model_step, 'feature_importances_'):
    importances = model_step.feature_importances_
    feat_names  = features_df.columns.tolist()

    fig, ax = plt.subplots(figsize=(8, 4))
    sorted_idx = np.argsort(importances)
    ax.barh([feat_names[i] for i in sorted_idx], importances[sorted_idx], color='teal')
    ax.set_title('Importance des features', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{DATA_DIR}/feature_importance.png', dpi=150)
    plt.show()
else:
    print('Feature importance non disponible pour ce modèle (Ridge).')

In [ ]:
# Sauvegarder
df_scored.to_csv(f'{DATA_DIR}/offres_ml_scored.csv', index=False)
np.save(f'{DATA_DIR}/features.npy', features_df.values)
print('Fichiers sauvegardés.')
print(f'  {DATA_DIR}/offres_ml_scored.csv')
print(f'  {DATA_DIR}/features.npy')
print(f'  ../models/best_scorer.pkl')
print('\nPour visualiser MLflow :')
print('  cd jobfit_phase0 && mlflow ui --backend-store-uri mlruns')
print('  Ouvrir http://localhost:5000')